# AI Interviewer - Gradio UI (Kaggle, GPU T4 x2, Internet ON)

**Attach as inputs (+ Add Input):** (1) the 5 datasets `phase 2 3 session 1..5 result`, (2) the `216 Video Model Runned (Acc:50%)` dataset, (3) optional: your sample interview video (used for a 12 s warm-up so the first real upload is fast).
The two `phase 1` datasets (3 GB) are training checkpoints and are **not** needed.

**Run the cells top to bottom in an interactive session** (not 'Save & Run All'). The last cell prints a public `gradio.live` link and keeps running - leave it running while you use the UI.

In [ ]:
!mkdir -p /kaggle/working/stages
!pip install -q -U transformers gradio
!pip install -q reportlab timm einops
# skeleton branch needs mediapipe's legacy pose API; the UI header shows ON/OFF after start-up
!pip install -q mediapipe || echo 'mediapipe install failed - skeleton branch will use neutral values'

In [ ]:
%%writefile /kaggle/working/stages/s0_config.py
import os, sys, glob, json, math, time, re, copy, random, subprocess, tempfile, warnings, datetime, io
warnings.filterwarnings("ignore")
import numpy as np, cv2
import torch, torch.nn as nn, torch.nn.functional as F

# =============================== EDIT THESE =================================
VIDEO_PATH  = "/kaggle/input/datasets/bdivyalakshmi/new-interview-video/Mr. John Interview.mp4"   # e.g. "/kaggle/input/my-interview/candidate.mp4"; None -> first video found in /kaggle/input
SPEECH_CKPTS = None  # list of speech .pt files. None -> AUTO-FIND every phase3_Ses0*.pt (5 sessions x F/M = 10) in all attached inputs
EMB_FROM    = 0      # which loaded speech model's 128-d embedding is passed to the fusion step (index in the sorted list)
VIDEO_CKPT  = None   # the "62 videos" final_model_*.pt ; None -> auto-search
FUSION_CKPT = None   # optional: a trained fusion .pt (none exists yet -> prior-initialised, see notes at the end)

CANDIDATE_NAME = None   # shown in the report header; None -> taken from the video file name
WINDOW_SEC  = 15     # one report entry per 15 seconds
SUBSEG_SEC  = 3      # each 15 s window is analysed as 5 time-steps of 3 s (the fusion model aligns these)
SR          = 16000
N_SKEL_FRAMES = 64   # frames per clip for the skeleton / Swin branches (same as training cache: 64)
SILENCE_RMS = 0.008  # below this RMS a 3 s step is treated as "no speech"
REPORT_NAME = "Candidate_Interview_Evaluation_Report.pdf"
OUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.abspath("work")
# =============================================================================
os.makedirs(OUT_DIR, exist_ok=True)
random.seed(42); np.random.seed(42); torch.manual_seed(42)

n_gpu = torch.cuda.device_count()
DEV_S = torch.device("cuda:0" if n_gpu > 0 else "cpu")             # speech model
DEV_V = torch.device("cuda:1" if n_gpu > 1 else DEV_S)             # body-language model (2nd T4 if available)
print(f"GPUs: {n_gpu} | speech -> {DEV_S} | body-language -> {DEV_V}")

def find_files(patterns, roots=("/kaggle/input", "/kaggle/working", "/mnt/user-data/uploads")):
    out = []
    for r in roots:
        for p in patterns:
            out += glob.glob(os.path.join(r, "**", p), recursive=True)
    return sorted(set(out))

if VIDEO_PATH is None:
    vids = find_files(["*.mp4", "*.mov", "*.avi", "*.mkv", "*.webm"])
    VIDEO_PATH = vids[0] if vids else None
if SPEECH_CKPTS is None:
    SPEECH_CKPTS = find_files(["phase3_Ses0*.pt"])                       # your LOSO fold checkpoints: phase3_Ses01F.pt, phase3_Ses01M.pt ... phase3_Ses05M.pt
    if not SPEECH_CKPTS:                                                 # fallback names
        SPEECH_CKPTS = find_files(["ser_model_trained*.pt", "*phase3*.pt"])
SPEECH_CKPTS = sorted(set(SPEECH_CKPTS))
if VIDEO_CKPT is None:
    c = find_files(["final_model_*.pt"])
    VIDEO_CKPT = c[0] if c else None
print(f"SPEECH_CKPTS -> {len(SPEECH_CKPTS)} file(s) found")
for p in SPEECH_CKPTS: print("    ", p)
if len(SPEECH_CKPTS) == 0:
    print("   !! No speech checkpoints found. Click '+ Add Input' and attach your 5 session result datasets, or set SPEECH_CKPTS above.")
elif len(SPEECH_CKPTS) != 10:
    print(f"   (note: expected 10 = 5 sessions x F/M, found {len(SPEECH_CKPTS)} - make sure all 5 session datasets are attached)")
for k, v in [("VIDEO_PATH", VIDEO_PATH), ("VIDEO_CKPT", VIDEO_CKPT)]:
    print(f"{k:12s} -> {v}")
    if v is None or not os.path.exists(v):
        print(f"   !! {k} not found. Click '+ Add Input' and attach it, or set {k} above.")

In [ ]:
%%writefile /kaggle/working/stages/s1_split.py
# ===================== STEP 1 + 2 : read the video and split into AUDIO + VIDEO =====================
from scipy.io import wavfile
assert VIDEO_PATH and os.path.exists(VIDEO_PATH), "Set VIDEO_PATH (attach your interview video as a Kaggle input)."

cap = cv2.VideoCapture(VIDEO_PATH)
FPS = cap.get(cv2.CAP_PROP_FPS) or 25.0
N_FRAMES = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
VID_W, VID_H = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
DURATION = N_FRAMES / FPS
cap.release()
print(f"Video: {os.path.basename(VIDEO_PATH)} | {DURATION:.1f} s | {FPS:.1f} fps | {N_FRAMES} frames")

AUDIO_WAV = os.path.join(OUT_DIR, "candidate_audio_16k.wav")
VIDEO_ONLY = os.path.join(OUT_DIR, "candidate_video_only.mp4")
r1 = subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", VIDEO_PATH, "-vn", "-ac", "1", "-ar", str(SR),
                     "-acodec", "pcm_s16le", AUDIO_WAV], capture_output=True, text=True)
r2 = subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", VIDEO_PATH, "-an", "-c:v", "copy", VIDEO_ONLY],
                    capture_output=True, text=True)
HAS_AUDIO = r1.returncode == 0 and os.path.exists(AUDIO_WAV)
if HAS_AUDIO:
    _sr, _a = wavfile.read(AUDIO_WAV)
    AUDIO = (_a.astype(np.float32) / 32768.0) if _a.dtype == np.int16 else _a.astype(np.float32)
    print(f"Audio  -> {AUDIO_WAV}  ({len(AUDIO)/SR:.1f} s)")
else:
    AUDIO = np.zeros(int(DURATION * SR), dtype=np.float32)
    print("!! No audio track found - every step will be treated as 'no speech'.")
VIDEO_FOR_FRAMES = VIDEO_ONLY if (r2.returncode == 0 and os.path.exists(VIDEO_ONLY)) else VIDEO_PATH
print(f"Video-only stream -> {VIDEO_FOR_FRAMES}")

# ---- 15 s windows, each cut into SUBSEG_SEC time-steps ----
SEGS, WINDOWS = [], []
n_win = int(math.ceil(DURATION / WINDOW_SEC))
for w in range(n_win):
    w0, w1 = w * WINDOW_SEC, min((w + 1) * WINDOW_SEC, DURATION)
    ids = []
    t = w0
    while t < w1 - 1e-6:
        t1 = min(t + SUBSEG_SEC, w1)
        if t1 - t >= 1.5 or not ids:               # drop tiny tail pieces (<1.5 s) unless it is the only step
            SEGS.append(dict(win=w, t0=t, t1=t1)); ids.append(len(SEGS) - 1)
        elif ids:
            SEGS[ids[-1]]["t1"] = t1               # merge a short tail into the previous step
        t += SUBSEG_SEC
    WINDOWS.append(dict(idx=w, t0=w0, t1=w1, seg_ids=ids))
print(f"{len(WINDOWS)} windows of {WINDOW_SEC}s -> {len(SEGS)} time-steps of ~{SUBSEG_SEC}s")

In [ ]:
%%writefile /kaggle/working/stages/s2_speech.py
# ===================== STEP 3 : SPEECH EMOTION MODEL (audio only) =====================
# Same architecture as your all-session phase 2&3 notebook (Spatial CNN + Wav2Vec2 temporal + multi-level fusion + classifier).
import torchaudio
from transformers import Wav2Vec2Model, Wav2Vec2Config

class ParallelConvBranch(nn.Module):
    def __init__(self, cin, cout, k):
        super().__init__()
        if isinstance(k, int): k = (k, k)
        self.conv = nn.Conv2d(cin, cout, k, padding=(k[0] // 2, k[1] // 2)); self.bn = nn.BatchNorm2d(cout)
        self.relu = nn.ReLU(inplace=True); self.downsample = nn.AvgPool2d(2, 2, ceil_mode=True)
    def forward(self, x): return self.downsample(self.relu(self.bn(self.conv(x))))
class DeepConvBlock(nn.Module):
    def __init__(self, cin, cout, down):
        super().__init__()
        self.conv = nn.Conv2d(cin, cout, 3, padding=1); self.bn = nn.BatchNorm2d(cout); self.relu = nn.ReLU(inplace=True)
        self.downsample = nn.AvgPool2d(2, 2, ceil_mode=True) if down else None
    def forward(self, x):
        x = self.relu(self.bn(self.conv(x)))
        return self.downsample(x) if self.downsample is not None else x
class SpatialCNN(nn.Module):
    def __init__(self, n_mels=40, bc=8, final_dim=96):
        super().__init__()
        self.branch_a = ParallelConvBranch(1, bc, (11, 1)); self.branch_b = ParallelConvBranch(1, bc, (1, 9)); self.branch_c = ParallelConvBranch(1, bc, (5, 5))
        steps = [bc * 3, 32, 48, 64, 80, final_dim]
        self.deep_blocks = nn.Sequential(*[DeepConvBlock(steps[i], steps[i + 1], i < 4) for i in range(5)])
        self.gap = nn.AdaptiveAvgPool2d(1)
    def forward(self, x):
        xc = torch.cat([self.branch_a(x), self.branch_b(x), self.branch_c(x)], 1)
        return self.gap(self.deep_blocks(xc)).flatten(1)
class TemporalWav2Vec2(nn.Module):
    def __init__(self):
        super().__init__()
        self.wav2vec2 = Wav2Vec2Model(Wav2Vec2Config())      # wav2vec2-base architecture; weights come from YOUR checkpoint
        self.attn_proj = nn.Linear(self.wav2vec2.config.hidden_size, 1)
    @staticmethod
    def _norm(w, eps=1e-3, clamp=10.0):
        w = torch.nan_to_num(w, nan=0.0, posinf=0.0, neginf=0.0)
        m = w.mean(-1, keepdim=True); s = w.std(-1, keepdim=True).clamp(min=eps)
        return ((w - m) / s).clamp(-clamp, clamp)
    def forward(self, wav, attention_mask=None):
        F2 = self.wav2vec2(self._norm(wav)).last_hidden_state
        a = torch.softmax(self.attn_proj(F2).squeeze(-1), -1)
        return torch.bmm(a.unsqueeze(1), F2).squeeze(1)
class MultiLevelFusion(nn.Module):
    def __init__(self, sd=96, td=768, h1=512, h2=256, fd=128):
        super().__init__()
        self.fc1 = nn.Linear(sd + td, h1); self.bn1 = nn.BatchNorm1d(h1); self.fc2 = nn.Linear(h1, h2); self.bn2 = nn.BatchNorm1d(h2)
        self.fc3 = nn.Linear(h2, fd); self.relu = nn.ReLU(inplace=True)
        self.linear_s = nn.Linear(sd, fd); self.linear_t = nn.Linear(td, fd); self.norm = nn.LayerNorm(fd)
    def forward(self, Xs, Ft):
        h = self.relu(self.bn1(self.fc1(torch.cat([Xs, Ft], -1)))); h = self.relu(self.bn2(self.fc2(h)))
        return self.norm(self.linear_s(Xs) + self.linear_t(Ft) + self.fc3(h))
class EmotionClassifier(nn.Module):
    def __init__(self, fd=128, nc=6):
        super().__init__(); self.classifier = nn.Linear(fd, nc)
    def forward(self, x): return torch.softmax(self.classifier(x), -1)
class SERModel(nn.Module):
    EMOTION_CLASSES = ["anger", "happiness", "sadness", "neutral", "frustration", "fear"]
    def __init__(self):
        super().__init__()
        self.spatial = SpatialCNN(); self.temporal = TemporalWav2Vec2()
        self.fusion = MultiLevelFusion(); self.classifier = EmotionClassifier(128, len(self.EMOTION_CLASSES))
    def forward(self, wav, fbank):
        Xs = self.spatial(fbank); Ft = self.temporal(wav); F_st = self.fusion(Xs, Ft)
        return {"probs": self.classifier(F_st), "emb": F_st}

def _remap_pos_conv(sd, to_new):
    out = {}
    for k, v in sd.items():
        if to_new:  k = k.replace("conv.weight_g", "conv.parametrizations.weight.original0").replace("conv.weight_v", "conv.parametrizations.weight.original1")
        else:       k = k.replace("conv.parametrizations.weight.original0", "conv.weight_g").replace("conv.parametrizations.weight.original1", "conv.weight_v")
        out[k] = v
    return out

def load_speech_model(path, dev):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    if isinstance(ck, dict):
        for k in ("model_state", "state_dict", "model_state_dict"):
            if k in ck and isinstance(ck[k], dict): ck = ck[k]; break
    sd = {k.replace("module.", "", 1): v for k, v in ck.items()}
    m = SERModel()
    for attempt in (sd, _remap_pos_conv(sd, True), _remap_pos_conv(sd, False)):
        try:
            m.load_state_dict(attempt, strict=True); break
        except RuntimeError as e:
            last = e
    else:
        raise RuntimeError(f"Speech checkpoint does not match the SERModel layout:\n{last}")
    return m.to(dev).eval()

def make_fbank(wav_1d, n_mels=40):
    # Log-Mel filterbank, shape (1, frames, 40). NOTE: mirrors a standard 25ms/10ms Kaldi fbank + mean normalisation;
    # if your preprocessing_ser.py used different settings, change ONLY this function.
    fb = torchaudio.compliance.kaldi.fbank(wav_1d.unsqueeze(0), num_mel_bins=n_mels, frame_length=25.0, frame_shift=10.0,
                                           dither=0.0, sample_frequency=SR)
    fb = fb - fb.mean(0, keepdim=True)
    return fb.unsqueeze(0)

SPEECH_CLASSES = SERModel.EMOTION_CLASSES
SPEECH_OUT = None
SPEECH_MODELS_USED = []                                  # paths that loaded successfully
if SPEECH_CKPTS:
    sers = []
    for p in SPEECH_CKPTS:
        try:
            sers.append(load_speech_model(p, DEV_S)); SPEECH_MODELS_USED.append(p)
            print(f"  loaded {os.path.basename(p)}")
        except Exception as e:
            print(f"  !! skipped {p}: {str(e)[:200]}")
    assert sers, "None of the speech checkpoints could be loaded - see the messages above."
    EMB_IDX = min(EMB_FROM, len(sers) - 1)
    print(f"Speech ensemble: {len(sers)} models | embedding for fusion from model #{EMB_IDX} ({os.path.basename(SPEECH_MODELS_USED[EMB_IDX])})")
    SPEECH_OUT = []
    t0 = time.time()
    with torch.no_grad():
        for i, sg in enumerate(SEGS):
            a = AUDIO[int(sg["t0"] * SR): int(sg["t1"] * SR)]
            rms = float(np.sqrt(np.mean(a ** 2))) if len(a) else 0.0
            if len(a) < int(1.0 * SR) or rms < SILENCE_RMS:
                SPEECH_OUT.append(dict(silent=True, probs=np.full(6, 1 / 6, np.float32), emb=np.zeros(128, np.float32), rms=rms)); continue
            wav = torch.from_numpy(a).float()
            fbank = make_fbank(wav).to(DEV_S).unsqueeze(0)          # (1, 1, frames, 40)
            wav_b = wav.unsqueeze(0).to(DEV_S)
            outs = [m(wav_b, fbank) for m in sers]                  # one forward pass per session/fold model
            probs_all = np.stack([o["probs"][0].cpu().numpy() for o in outs])     # (n_models, 6)
            SPEECH_OUT.append(dict(silent=False, probs=probs_all.mean(0).astype(np.float32),     # ensemble = average of all models
                                   emb=outs[EMB_IDX]["emb"][0].cpu().numpy(), rms=rms, probs_per_model=probs_all))
            if (i + 1) % 20 == 0: print(f"  speech {i+1}/{len(SEGS)}")
    print(f"Speech inference done in {time.time()-t0:.1f}s | speech steps: {sum(not o['silent'] for o in SPEECH_OUT)}/{len(SEGS)}")
    del sers; torch.cuda.empty_cache()
else:
    print("!! No speech checkpoints found - attach the 5 session datasets (or set SPEECH_CKPTS) and re-run this cell.")


In [ ]:
%%writefile /kaggle/working/stages/s3_body_load.py
# ===================== STEP 4 : BODY-LANGUAGE MODEL (video only) =====================
# Rebuilds your "62 videos" model: fine-tuned Video-Swin (RGB + DCT) + LaViLa + 3-stream skeleton transformer + fusion head.
import torchvision
from torchvision.models import video as tvv
VCK = torch.load(VIDEO_CKPT, map_location="cpu", weights_only=False) if VIDEO_CKPT and os.path.exists(VIDEO_CKPT) else None
assert VCK is not None, "Set VIDEO_CKPT to your final_model_*.pt from the 62-video notebook."
BODY_CLASSES = list(VCK["classes"]); NUM_BODY = len(BODY_CLASSES); MODS = list(VCK["modalities"])
FEAT_DIMS, STREAM_DIMS, STATS = VCK["feat_dims"], VCK["stream_dims"], VCK["stats"]
print(f"Body-language model: variant='{VCK['variant']}' | modalities={MODS} | {NUM_BODY} classes | CV held-out acc={VCK.get('cv_heldout_acc', float('nan')):.1%}")
print("Classes:", BODY_CLASSES)

# ---------- head (identical to training) ----------
HEAD_INPUT_DROPOUT, HEAD_DROPOUT = 0.3, 0.5
def _enc_block(dim, n_heads=2, ff=128, dr=0.2):
    return nn.TransformerEncoderLayer(d_model=dim, nhead=n_heads, dim_feedforward=ff, dropout=dr, batch_first=True, activation="relu")
class FeatureProjector(nn.Module):
    def __init__(self, i, o=64): super().__init__(); self.drop = nn.Dropout(HEAD_INPUT_DROPOUT); self.proj = nn.Linear(i, o)
    def forward(self, x): return F.relu(self.proj(self.drop(x)))
class TokenTransformerHead(nn.Module):
    def __init__(self, i, n_tokens, d=64):
        super().__init__(); self.drop = nn.Dropout(HEAD_INPUT_DROPOUT); self.embed = nn.Linear(i, d)
        self.pos_encoding = nn.Parameter(torch.zeros(1, n_tokens, d)); self.encoder = _enc_block(d)
    def forward(self, x): return self.encoder(self.embed(self.drop(x)) + self.pos_encoding[:, :x.shape[1], :]).mean(1)
class SkeletonStreamEncoder(nn.Module):
    def __init__(self, i, L, d=64):
        super().__init__(); self.embed = nn.Linear(i, d); self.pos_encoding = nn.Parameter(torch.zeros(1, L, d)); self.encoder = _enc_block(d)
    def forward(self, x): return self.encoder(self.embed(x) + self.pos_encoding[:, :x.shape[1], :]).mean(1)
class ThreeStreamSkeletonTransformer(nn.Module):
    def __init__(self, dims, d=64):
        super().__init__(); (dp, lp), (dt, lt), (ds, ls) = dims["P"], dims["T"], dims["S"]
        self.pos_enc = SkeletonStreamEncoder(dp, lp, d); self.tmp_enc = SkeletonStreamEncoder(dt, lt, d); self.spa_enc = SkeletonStreamEncoder(ds, ls, d)
    def forward(self, P, T, S): return self.pos_enc(P) + self.tmp_enc(T) + self.spa_enc(S)
class FusionClassifier(nn.Module):
    def __init__(self, i, nc, dr=HEAD_DROPOUT):
        super().__init__(); self.head = nn.Sequential(nn.Linear(i, 64), nn.ReLU(), nn.Dropout(dr), nn.Linear(64, nc))
    def forward(self, x): return self.head(x)
class BodyLangModel(nn.Module):
    def __init__(self, modalities, nc):
        super().__init__(); self.modalities = list(modalities)
        if "rgb" in modalities: self.rgb_enc = FeatureProjector(FEAT_DIMS["rgb"])
        if "dct" in modalities: self.dct_enc = FeatureProjector(FEAT_DIMS["dct"])
        if "skel" in modalities: self.skel_enc = ThreeStreamSkeletonTransformer(STREAM_DIMS)
        if "lavila" in modalities: self.lavila_enc = TokenTransformerHead(FEAT_DIMS["lavila"][1], FEAT_DIMS["lavila"][0])
        self.classifier = FusionClassifier(64 * len(modalities), nc)
    def forward(self, rgb, dct, P, T, S, lav):
        f = []
        if "rgb" in self.modalities: f.append(self.rgb_enc(rgb))
        if "dct" in self.modalities: f.append(self.dct_enc(dct))
        if "skel" in self.modalities: f.append(self.skel_enc(P, T, S))
        if "lavila" in self.modalities: f.append(self.lavila_enc(lav))
        emb = torch.cat(f, 1)
        return F.softmax(self.classifier(emb), 1), emb

head = BodyLangModel(MODS, NUM_BODY); head.load_state_dict(VCK["head_state"]); head = head.to(DEV_V).eval()

# ---------- Video-Swin backbones (pretrained Kinetics weights + YOUR fine-tuned layers) ----------
def build_backbone():
    for name, w in (("swin3d_b", "KINETICS400_IMAGENET22K_V1"), ("swin3d_b", "KINETICS400_V1"), ("swin3d_t", "KINETICS400_V1")):
        try:
            cls = {"swin3d_b": "Swin3D_B_Weights", "swin3d_t": "Swin3D_T_Weights"}[name]
            m = getattr(tvv, name)(weights=getattr(getattr(tvv, cls), w)); dim = m.head.in_features; m.head = nn.Identity()
            print(f"  backbone {name}/{w} (dim {dim})"); return m, dim
        except Exception as e:
            print(f"  could not load {name}/{w}: {e!r}")
    raise RuntimeError("Could not download Video-Swin weights - turn Internet ON in the Kaggle notebook settings.")
def make_finetuned(trained):
    bb, dim = build_backbone(); params = dict(bb.named_parameters())
    miss = [k for k in trained if k not in params]
    assert not miss, f"fine-tuned parameter names do not match the backbone: {miss[:3]}"
    with torch.no_grad():
        for k, v in trained.items(): params[k].copy_(v.float())
    for p in bb.parameters(): p.requires_grad = False
    return bb.to(DEV_V).eval(), dim
BB = {}
for mod in ("rgb", "dct"):
    if mod in MODS:
        BB[mod], d_ = make_finetuned(VCK[f"{mod}_finetuned_params"]); assert d_ == FEAT_DIMS[mod]

# ---------- LaViLa (only if the winning variant used it) ----------
LAV = None
if "lavila" in MODS:
    try:
        import timm, einops, urllib.request
        tmp = tempfile.gettempdir(); src = os.path.join(tmp, "lavila_src")
        raw = "https://raw.githubusercontent.com/facebookresearch/LaViLa/main/"
        for rel in ("lavila/models/timesformer.py", "lavila/models/openai_model.py"):
            dest = os.path.join(src, rel)
            if not os.path.isfile(dest):
                os.makedirs(os.path.dirname(dest), exist_ok=True); urllib.request.urlretrieve(raw + rel, dest)
        sys.path.insert(0, src)
        try:
            from timm.models.layers import DropPath, to_2tuple, trunc_normal_
        except Exception:
            import types, timm.layers as tl
            shim = types.ModuleType("timm.models.layers"); shim.DropPath, shim.to_2tuple, shim.trunc_normal_ = tl.DropPath, tl.to_2tuple, tl.trunc_normal_
            sys.modules["timm.models.layers"] = shim
        from lavila.models.timesformer import SpaceTimeTransformer
        from lavila.models.openai_model import QuickGELU
        ck_path = (find_files(["*timesformer*.pth"]) or [None])[0]
        if ck_path is None:
            ck_path = os.path.join(tmp, "lavila_tsf.pth")
            urllib.request.urlretrieve("https://dl.fbaipublicfiles.com/lavila/checkpoints/dual_encoders/ego4d/clip_openai_timesformer_base.narrator_rephraser.ep_0005.md5sum_d73a9c.pth", ck_path)
        LAV = SpaceTimeTransformer(num_frames=4, time_init="zeros", attention_style="frozen-in-time", ln_pre=True, act_layer=QuickGELU, is_tanh_gating=False, drop_path_rate=0.0)
        LAV.head, LAV.pre_logits, LAV.fc = nn.Identity(), nn.Identity(), nn.Identity()
        ck = torch.load(ck_path, map_location="cpu", weights_only=False); ck = ck.get("state_dict", ck)
        sd = {k.replace("module.", "")[len("visual."):]: v for k, v in ck.items() if k.replace("module.", "").startswith("visual.")}
        r = LAV.load_state_dict(sd, strict=False); assert not r.missing_keys, r.missing_keys[:3]
        LAV = LAV.to(DEV_V).eval(); print("LaViLa encoder ready.")
    except Exception as e:
        print(f"!! LaViLa unavailable ({e!r}) - that branch will be fed neutral (z = 0) values; accuracy will drop.")
        LAV = None

# ---------- MediaPipe pose (skeleton stream) ----------
POSE = None
if "skel" in MODS:
    try:
        import mediapipe as mp
        POSE_CLS = mp.solutions.pose.Pose
        print("MediaPipe Pose ready.")
    except Exception as e:
        POSE_CLS = None; print(f"!! MediaPipe unavailable ({e!r}) - skeleton branch will be fed neutral values.")

In [ ]:
%%writefile /kaggle/working/stages/s4_body_run.py
# ===================== body-language inference on every time-step =====================
KIN_MEAN = torch.tensor([0.4850, 0.4560, 0.4060]).view(1, 3, 1, 1); KIN_STD = torch.tensor([0.2290, 0.2240, 0.2250]).view(1, 3, 1, 1)
LAV_MEAN = (torch.tensor([108.3272985, 116.7460125, 104.09373615]) / 255.0).view(1, 3, 1, 1)
LAV_STD = (torch.tensor([68.5005327, 66.6321579, 70.32316305]) / 255.0).view(1, 3, 1, 1)

def prep_swin(frames_u8, n=32, size=224):                      # (T,H,W,3) uint8 -> (1,3,n,size,size)
    x = torch.as_tensor(frames_u8).float() / 255.0
    x = x[torch.linspace(0, x.shape[0] - 1, n).round().long()].permute(0, 3, 1, 2)
    x = F.interpolate(x, size=(size, size), mode="bilinear", align_corners=False)
    return ((x - KIN_MEAN) / KIN_STD).permute(1, 0, 2, 3).unsqueeze(0).contiguous()
def prep_lavila(frames_u8, n_samples=8, n_seg=4, size=224):    # -> (8,3,4,224,224)
    x = torch.as_tensor(frames_u8).float() / 255.0; T = x.shape[0]; b = np.linspace(0, T, n_seg + 1).astype(int); out = []
    for _ in range(n_samples):
        idx = [random.randint(int(b[i]), max(int(b[i]), int(b[i + 1]) - 1)) for i in range(n_seg)]
        v = F.interpolate(x[idx].permute(0, 3, 1, 2), size=(size, size), mode="bilinear", align_corners=False)
        out.append(((v - LAV_MEAN) / LAV_STD).permute(1, 0, 2, 3))
    return torch.stack(out)

def make_dct_frames(frames_u8):
    # Frame-wise frequency-domain view of the RGB frames (per-channel 2-D DCT, log-magnitude, rescaled to uint8).
    # NOTE: written from the architecture description - if your preprocessing notebook used a different DCT recipe,
    # change ONLY this function so DCT inputs match what the model was trained on.
    out = np.zeros_like(frames_u8)
    for t in range(frames_u8.shape[0]):
        for c in range(3):
            d = cv2.dct(frames_u8[t, :, :, c].astype(np.float32) / 255.0)
            d = np.log1p(np.abs(d)); d = (d - d.min()) / (d.max() - d.min() + 1e-8)
            out[t, :, :, c] = (d * 255).astype(np.uint8)
    return out

def skeleton_streams(frames_u8):
    # P: (64,66) x,y of 33 MediaPipe joints | T: (63,66) frame-to-frame motion | S: (64,62) the 31 non-hip joints relative to hip-centre
    # NOTE: S is reconstructed from its dimension (31 joints x 2); verify against your preprocessing notebook if results look off.
    L = frames_u8.shape[0]; P = np.zeros((L, 33, 2), np.float32); last = None; found = 0
    if POSE_CLS is not None:
        with POSE_CLS(static_image_mode=False, model_complexity=1) as pose:
            for t in range(L):
                res = pose.process(frames_u8[t])
                if res.pose_landmarks:
                    last = np.array([[p.x, p.y] for p in res.pose_landmarks.landmark], np.float32); found += 1
                if last is not None: P[t] = last
    Pf = P.reshape(L, -1); Tm = np.diff(Pf, axis=0)
    hip = (P[:, 23:24, :] + P[:, 24:25, :]) / 2.0
    S = (P - hip)[:, [i for i in range(33) if i not in (23, 24)], :].reshape(L, -1)
    return Pf, Tm, S, found / max(L, 1)

def stdz(x, key):
    mu, sd = STATS[key]; return ((x.to(mu.device) - mu) / sd)
def entropy_conf(p):
    p = np.clip(p, 1e-8, 1); return float(1.0 - (-(p * np.log(p)).sum()) / np.log(len(p)))

@torch.no_grad()
def body_step(frames):                                         # frames: (64,224,224,3) uint8
    amp = dict(device_type="cuda", dtype=torch.float16, enabled=DEV_V.type == "cuda")
    rgb = dct = P = T = S = lav = None; info = {}
    if "rgb" in MODS:
        with torch.autocast(**amp): f = BB["rgb"](prep_swin(frames).to(DEV_V).float())
        rgb = stdz(torch.nan_to_num(f.float()).cpu().unsqueeze(1), "rgb")[:, 0].to(DEV_V)
    if "dct" in MODS:
        with torch.autocast(**amp): f = BB["dct"](prep_swin(make_dct_frames(frames)).to(DEV_V).float())
        dct = stdz(torch.nan_to_num(f.float()).cpu().unsqueeze(1), "dct")[:, 0].to(DEV_V)
    if "skel" in MODS:
        Pn, Tn, Sn, det = skeleton_streams(frames); info["pose_detect_rate"] = det
        if det == 0 or POSE_CLS is None:
            P = torch.zeros(1, STREAM_DIMS["P"][1], STREAM_DIMS["P"][0], device=DEV_V)
            T = torch.zeros(1, STREAM_DIMS["T"][1], STREAM_DIMS["T"][0], device=DEV_V)
            S = torch.zeros(1, STREAM_DIMS["S"][1], STREAM_DIMS["S"][0], device=DEV_V)          # z-score 0 = the training mean
        else:
            def fit(x, key):
                L, D = STREAM_DIMS[key][1], STREAM_DIMS[key][0]
                assert x.shape[1] == D, f"skeleton stream {key}: built width {x.shape[1]} but the model expects {D} - adapt skeleton_streams()"
                x = x[:L] if x.shape[0] >= L else np.vstack([x, np.repeat(x[-1:], L - x.shape[0], 0)])
                return stdz(torch.from_numpy(x).float().unsqueeze(0), key).to(DEV_V)
            P, T, S = fit(Pn, "P"), fit(Tn, "T"), fit(Sn, "S")
    if "lavila" in MODS:
        if LAV is not None:
            with torch.autocast(**amp): tok = LAV(prep_lavila(frames).to(DEV_V).float())
            lav = stdz(torch.nan_to_num(tok.float()).cpu().unsqueeze(0), "lav").to(DEV_V)
        else:
            lav = torch.zeros(1, FEAT_DIMS["lavila"][0], FEAT_DIMS["lavila"][1], device=DEV_V)
    dummy = torch.zeros(1, 1, device=DEV_V)
    probs, emb = head(rgb if rgb is not None else dummy, dct if dct is not None else dummy, P, T, S, lav)
    return probs[0].float().cpu().numpy(), emb[0].float().cpu().numpy(), info

def stream_step_frames(path, segs, fps, n=N_SKEL_FRAMES, size=224):
    # one sequential decode pass; yields (64,224,224,3) uint8 frames + a thumbnail for each time-step
    cap = cv2.VideoCapture(path); cur = 0
    for sg in segs:
        f0 = int(round(sg["t0"] * fps)); f1 = max(f0 + 1, int(round(sg["t1"] * fps)))
        want = np.linspace(f0, f1 - 1, n).round().astype(int); need = set(want.tolist()); got = {}; mid = (f0 + f1) // 2; thumb = None
        while cur < f1:
            ok, fr = cap.read()
            if not ok: break
            if cur in need: got[cur] = cv2.resize(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB), (size, size))
            if cur == mid: thumb = cv2.cvtColor(cv2.resize(fr, (320, int(320 * fr.shape[0] / fr.shape[1]))), cv2.COLOR_BGR2RGB)
            cur += 1
        if not got: yield None, None; continue
        keys = sorted(got); seq = [got[min(keys, key=lambda k: abs(k - w))] for w in want]
        yield np.stack(seq), thumb
    cap.release()

BODY_OUT, THUMBS = [], []
t0 = time.time()
for i, (frames, thumb) in enumerate(stream_step_frames(VIDEO_FOR_FRAMES, SEGS, FPS)):
    if frames is None:
        BODY_OUT.append(dict(probs=np.full(NUM_BODY, 1.0 / NUM_BODY, np.float32), emb=np.zeros(64 * len(MODS), np.float32), valid=False)); THUMBS.append(None); continue
    p, e, info = body_step(frames)
    BODY_OUT.append(dict(probs=p, emb=e, valid=True, **info)); THUMBS.append(thumb)
    if (i + 1) % 10 == 0 or i + 1 == len(SEGS): print(f"  body-language {i+1}/{len(SEGS)}  ({time.time()-t0:.0f}s)")
for m in list(BB.values()) + [head] + ([LAV] if LAV is not None else []): m.cpu()
torch.cuda.empty_cache()
dr = [o.get("pose_detect_rate") for o in BODY_OUT if o.get("pose_detect_rate") is not None]
if dr: print(f"Pose detected in {np.mean(dr):.0%} of frames on average.")

In [ ]:
%%writefile /kaggle/working/stages/s5_fusion.py
# ===================== STEP 5 : CROSS-ATTENTION BASED EMBEDDING FUSION (new architecture) =====================
#   Temporal Alignment Layer  ->  Bidirectional Cross-Modal Attention Transformer  ->  Gated Multimodal Unit (GMU)
class TemporalAlignmentLayer(nn.Module):
    # projects speech / body-language step embeddings to a shared width, resamples both streams onto one common
    # time grid (identical if both have the same number of steps), and adds time + modality encodings.
    def __init__(self, ds_in, dv_in, d=128):
        super().__init__()
        self.ps = nn.Sequential(nn.LayerNorm(ds_in), nn.Linear(ds_in, d)); self.pv = nn.Sequential(nn.LayerNorm(dv_in), nn.Linear(dv_in, d))
        self.mod_s = nn.Parameter(torch.zeros(1, 1, d)); self.mod_v = nn.Parameter(torch.zeros(1, 1, d)); self.d = d
    @staticmethod
    def _resample(x, L):                                           # x: (B,L0,D) or (B,L0)
        sq = x.dim() == 2
        if sq: x = x.unsqueeze(-1)
        if x.shape[1] != L: x = F.interpolate(x.transpose(1, 2), size=L, mode="linear", align_corners=True).transpose(1, 2)
        return x.squeeze(-1) if sq else x
    def _time_enc(self, L, dev):
        pos = torch.linspace(0, 1, L, device=dev).unsqueeze(1) * 100.0; i = torch.arange(self.d // 2, device=dev).unsqueeze(0)
        ang = pos / torch.pow(10000.0, 2 * i / self.d); return torch.cat([ang.sin(), ang.cos()], -1).unsqueeze(0)
    def forward(self, xs, xv, ms, mv, cs, cv_):
        L = max(xs.shape[1], xv.shape[1]); te = self._time_enc(L, xs.device)
        s = self._resample(self.ps(xs), L) + te + self.mod_s; v = self._resample(self.pv(xv), L) + te + self.mod_v
        ms = self._resample(ms.float(), L) > 0.5; mv = self._resample(mv.float(), L) > 0.5
        return s, v, ms, mv, self._resample(cs, L), self._resample(cv_, L)
def _kpm(valid):                                                    # True = ignore; never mask a whole row
    k = ~valid; return k & ~k.all(1, keepdim=True)
class BiCrossAttnLayer(nn.Module):
    def __init__(self, d, heads, drop=0.1):
        super().__init__()
        self.s2v = nn.MultiheadAttention(d, heads, dropout=drop, batch_first=True); self.v2s = nn.MultiheadAttention(d, heads, dropout=drop, batch_first=True)
        self.ns1, self.nv1, self.ns2, self.nv2 = (nn.LayerNorm(d) for _ in range(4))
        self.ffs = nn.Sequential(nn.Linear(d, 2 * d), nn.GELU(), nn.Dropout(drop), nn.Linear(2 * d, d))
        self.ffv = nn.Sequential(nn.Linear(d, 2 * d), nn.GELU(), nn.Dropout(drop), nn.Linear(2 * d, d)); self.drop = nn.Dropout(drop)
    def forward(self, s, v, ms, mv):
        sa, a_sv = self.s2v(s, v, v, key_padding_mask=_kpm(mv), need_weights=True, average_attn_weights=True)   # speech queries -> video keys
        va, a_vs = self.v2s(v, s, s, key_padding_mask=_kpm(ms), need_weights=True, average_attn_weights=True)   # video queries -> speech keys
        s = self.ns1(s + self.drop(sa)); v = self.nv1(v + self.drop(va))
        return self.ns2(s + self.drop(self.ffs(s))), self.nv2(v + self.drop(self.ffv(v))), a_sv, a_vs
class GatedMultimodalUnit(nn.Module):
    # h = z * tanh(Ws s) + (1 - z) * tanh(Wv v);  z = sigmoid(Wz [s;v] + k * (conf_speech - conf_video))
    # Wz starts at ZERO, so before any training the gate is driven purely by each modality's confidence (transparent prior).
    def __init__(self, d, conf_scale=4.0):
        super().__init__(); self.ws = nn.Linear(d, d); self.wv = nn.Linear(d, d); self.wz = nn.Linear(2 * d, 1); self.k = conf_scale
        nn.init.zeros_(self.wz.weight); nn.init.zeros_(self.wz.bias)
    def forward(self, s, v, cs, cv_):
        z = torch.sigmoid(self.wz(torch.cat([s, v], -1)).squeeze(-1) + self.k * (cs - cv_))              # (B,L) = weight of SPEECH
        h = z.unsqueeze(-1) * torch.tanh(self.ws(s)) + (1 - z.unsqueeze(-1)) * torch.tanh(self.wv(v))
        return h, z
class CrossAttentionEmbeddingFusion(nn.Module):
    def __init__(self, ds_in, dv_in, d=128, heads=4, n_layers=2):
        super().__init__()
        self.align = TemporalAlignmentLayer(ds_in, dv_in, d); self.layers = nn.ModuleList([BiCrossAttnLayer(d, heads) for _ in range(n_layers)])
        self.gmu = GatedMultimodalUnit(d); self.pool = nn.Linear(d, 1)
    def forward(self, xs, xv, ms, mv, cs, cv_):
        s, v, ms, mv, cs, cv_ = self.align(xs, xv, ms, mv, cs, cv_)
        for layer in self.layers: s, v, a_sv, a_vs = layer(s, v, ms, mv)
        h, z = self.gmu(s, v, cs, cv_); w = torch.softmax(self.pool(h).squeeze(-1), -1)
        return dict(fused=(w.unsqueeze(-1) * h).sum(1), steps=h, gate=z, attn_s2v=a_sv, attn_v2s=a_vs, pool_w=w)

DS_IN, DV_IN = 128 + len(SPEECH_CLASSES), 64 * len(MODS) + NUM_BODY
torch.manual_seed(7)                                               # fixed seed -> reproducible prior-initialised weights
fusion = CrossAttentionEmbeddingFusion(DS_IN, DV_IN).to(DEV_S).eval()
if FUSION_CKPT and os.path.exists(FUSION_CKPT):
    fusion.load_state_dict(torch.load(FUSION_CKPT, map_location=DEV_S)); FUSION_TRAINED = True; print("Loaded trained fusion weights.")
else:
    FUSION_TRAINED = False
    print("Fusion weights: prior-initialised (no paired audio-visual labels exist yet to train them). "
          "The GMU gate is therefore confidence-driven; see the notes at the end.")

FUSION_OUT = []                                                    # one dict per 15 s window
with torch.no_grad():
    for w in WINDOWS:
        ids = w["seg_ids"]
        xs = torch.tensor(np.stack([np.concatenate([SPEECH_OUT[i]["emb"], SPEECH_OUT[i]["probs"]]) for i in ids]), dtype=torch.float32)[None].to(DEV_S)
        xv = torch.tensor(np.stack([np.concatenate([BODY_OUT[i]["emb"], BODY_OUT[i]["probs"]]) for i in ids]), dtype=torch.float32)[None].to(DEV_S)
        ms = torch.tensor([[not SPEECH_OUT[i]["silent"] for i in ids]]).to(DEV_S); mv = torch.tensor([[BODY_OUT[i]["valid"] for i in ids]]).to(DEV_S)
        cs = torch.tensor([[0.0 if SPEECH_OUT[i]["silent"] else entropy_conf(SPEECH_OUT[i]["probs"]) for i in ids]], dtype=torch.float32).to(DEV_S)
        cv_ = torch.tensor([[entropy_conf(BODY_OUT[i]["probs"]) if BODY_OUT[i]["valid"] else 0.0 for i in ids]], dtype=torch.float32).to(DEV_S)
        o = fusion(xs, xv, ms, mv, cs, cv_)
        FUSION_OUT.append(dict(fused=o["fused"][0].cpu().numpy(), gate=o["gate"][0].cpu().numpy(), attn_s2v=o["attn_s2v"][0].cpu().numpy(),
                               attn_v2s=o["attn_v2s"][0].cpu().numpy(), conf_s=cs[0].cpu().numpy(), conf_v=cv_[0].cpu().numpy()))
print(f"Fusion done for {len(FUSION_OUT)} windows. Example (window 0): speech weight per step = {np.round(FUSION_OUT[0]['gate'], 2)}")

In [ ]:
%%writefile /kaggle/working/stages/s6_scores_defs.py
# ===================== STEP 6a : turn model outputs into EXPLAINABLE per-window scores =====================
# Each emotion / body-language class is mapped to three interpretable traits (0-1): positivity, composure, engagement.
# EDIT these tables if you disagree with a mapping - the whole report recomputes from them.
SPEECH_TRAITS = {            # (positivity, composure, engagement)
    "anger": (0.20, 0.20, 0.70), "happiness": (1.00, 0.80, 0.90), "sadness": (0.20, 0.40, 0.30),
    "neutral": (0.55, 0.85, 0.50), "frustration": (0.20, 0.25, 0.60), "fear": (0.25, 0.15, 0.40)}
POS_KEYS = ["engage", "open", "smil", "confiden", "nod", "lean", "attentive", "happy", "relax", "calm", "gestur", "upright", "eye contact", "posit", "enthus", "interest"]
NEG_KEYS = ["fidget", "closed", "cross", "anxi", "nervous", "slouch", "avert", "look away", "looking away", "away", "hide", "touch", "tense", "fear", "angry", "anger", "sad", "bored", "stress", "frown", "withdraw", "defens", "disengag", "shrug", "fold"]
IDLE_KEYS = ["idle", "still", "neutral", "sit", "stand", "rest", "static"]
BODY_TRAIT_OVERRIDE = {}     # e.g. {"arms_crossed": (0.3, 0.4, 0.4)}  <- put your own (positivity, composure, engagement) here

def build_body_traits(classes):
    tab, auto = {}, []
    for c in classes:
        if c in BODY_TRAIT_OVERRIDE: tab[c] = BODY_TRAIT_OVERRIDE[c]; continue
        n = c.lower().replace("_", " ").replace("-", " ")
        if any(k in n for k in NEG_KEYS): tab[c] = (0.25, 0.25, 0.45)
        elif any(k in n for k in POS_KEYS): tab[c] = (0.80, 0.75, 0.75)
        elif any(k in n for k in IDLE_KEYS): tab[c] = (0.50, 0.70, 0.35)
        else: tab[c] = (0.50, 0.50, 0.50); auto.append(c)
    return tab, auto

CATEGORIES = [   # name, weight, what it looks at  (this replaces the template's "Category / Weighting / Questions" columns)
    ("Vocal Emotional Tone", 0.20, "Emotion in the voice (anger, happiness, sadness, neutral, frustration, fear) turned into tone positivity + steadiness."),
    ("Body-Language Presence", 0.20, "Posture, gesture and movement classes from the body-language model turned into positivity + steadiness."),
    ("Composure & Stability", 0.25, "Calm, steady delivery in voice and body versus stress / agitation cues."),
    ("Engagement & Energy", 0.20, "Attentiveness and energy in voice and body."),
    ("Audio-Visual Consistency", 0.15, "Whether voice and body language tell the same story (low agreement = mixed signals)."),
]
RATING_BANDS = [(7.5, "Excellent"), (6.0, "Good"), (4.5, "Fair"), (0.0, "Needs attention")]
def rating_for(x): return next(lbl for thr, lbl in RATING_BANDS if x >= thr)
def recommendation_for(x):
    if x >= 7.0: return "Positive behavioural signals - supports progressing the candidate (final decision stays with the interview panel)."
    if x >= 5.0: return "Mixed behavioural signals - review the flagged windows with the panel before deciding."
    return "Weak behavioural signals - recommend human review of the whole recording before any decision."

def analyse(windows, segs, speech_out, body_out, fusion_out, speech_classes, body_classes, body_traits):
    ST = np.array([SPEECH_TRAITS[c] for c in speech_classes], np.float32); BT = np.array([body_traits[c] for c in body_classes], np.float32)
    res = []
    for w, fo in zip(windows, fusion_out):
        ids = w["seg_ids"]; n = len(ids)
        sp = np.stack([speech_out[i]["probs"] for i in ids]); bp = np.stack([body_out[i]["probs"] for i in ids])
        silent = np.array([speech_out[i]["silent"] for i in ids]); valid = np.array([body_out[i]["valid"] for i in ids])
        z = fo["gate"].copy(); z[~valid] = 1.0; z[silent] = 0.0            # no speech -> speech carries no weight
        s_tr, b_tr = sp @ ST, bp @ BT                                        # (n,3) each
        f_tr = z[:, None] * s_tr + (1 - z[:, None]) * b_tr                   # gate-weighted fused traits
        both = (~silent) & valid
        cong = float(np.mean(1 - np.abs(s_tr[both] - b_tr[both]).mean(1))) if both.any() else None
        sp_act = sp[~silent]; speech_mean = sp_act.mean(0) if len(sp_act) else None
        body_mean = bp[valid].mean(0) if valid.any() else None
        cat = {}
        cat["Vocal Emotional Tone"] = float(s_tr[~silent][:, :2].mean()) if (~silent).any() else None
        cat["Body-Language Presence"] = float(b_tr[valid][:, :2].mean()) if valid.any() else None
        cat["Composure & Stability"] = float(f_tr[:, 1].mean())
        cat["Engagement & Energy"] = float(f_tr[:, 2].mean())
        cat["Audio-Visual Consistency"] = cong
        cat = {k: (None if v is None else round(10 * v, 1)) for k, v in cat.items()}
        ws = [(wt, cat[nm]) for nm, wt, _ in CATEGORIES if cat[nm] is not None]
        score = round(sum(wt * v for wt, v in ws) / sum(wt for wt, _ in ws), 1)
        res.append(dict(idx=w["idx"], t0=w["t0"], t1=w["t1"], n_steps=n, n_silent=int(silent.sum()), speech_mean=speech_mean, body_mean=body_mean,
                        speech_steps=sp, body_steps=bp, silent=silent, gate=z, cong=cong, cat=cat, score=score, rating=rating_for(score),
                        conf_s=float(fo["conf_s"][~silent].mean()) if (~silent).any() else None, conf_v=float(fo["conf_v"][valid].mean()) if valid.any() else None,
                        attn_s2v=fo["attn_s2v"], attn_v2s=fo["attn_v2s"], thumb=None))
    return res

def fmt_t(sec): return f"{int(sec // 60):02d}:{int(sec % 60):02d}"
def comment_for(r, speech_classes, body_classes):
    parts = []
    if r["speech_mean"] is not None:
        i = int(np.argmax(r["speech_mean"])); parts.append(f"Voice sounded mostly {speech_classes[i]} ({r['speech_mean'][i]:.0%}).")
    else: parts.append("No clear speech detected, so voice cues were not scored.")
    if r["body_mean"] is not None:
        order = np.argsort(r["body_mean"])[::-1]; a = order[0]; b = order[1] if len(order) > 1 else None
        s = f"Body language was mainly '{body_classes[a]}' ({r['body_mean'][a]:.0%})"
        if b is not None and r["body_mean"][b] > 0.15: s += f", then '{body_classes[b]}' ({r['body_mean'][b]:.0%})"
        parts.append(s + ".")
    if r["n_silent"] == r["n_steps"]: parts.append("Scored from body language only.")
    if r["cong"] is not None:
        parts.append("Voice and body cues agreed well." if r["cong"] >= 0.80 else "Voice and body gave somewhat mixed signals." if r["cong"] >= 0.65 else "Voice and body cues disagreed noticeably.")
    low = [k for k, v in (("voice", r["conf_s"]), ("body language", r["conf_v"])) if v is not None and v < 0.10]
    if low: parts.append(f"Model confidence for {' and '.join(low)} is low here, so treat this window as indicative only.")
    return " ".join(parts)

def overall_summary(res, speech_classes, body_classes):
    dur = np.array([r["t1"] - r["t0"] for r in res]); sc = np.array([r["score"] for r in res])
    overall = float(np.average(sc, weights=dur))
    catavg = {}
    for nm, wt, _ in CATEGORIES:
        v = [(r["cat"][nm], d) for r, d in zip(res, dur) if r["cat"][nm] is not None]
        catavg[nm] = round(float(np.average([a for a, _ in v], weights=[b for _, b in v])), 1) if v else None
    sp = [r["speech_mean"] for r in res if r["speech_mean"] is not None]; bd = [r["body_mean"] for r in res if r["body_mean"] is not None]
    order = np.argsort(sc)
    return dict(overall=round(overall, 1), rating=rating_for(overall), recommendation=recommendation_for(overall), cat=catavg,
                speech_share=np.mean(sp, 0) if sp else None, body_share=np.mean(bd, 0) if bd else None,
                best=[res[i]["idx"] for i in order[::-1][:2]], worst=[res[i]["idx"] for i in order[:2]],
                speech_fraction=float(1 - sum(r["n_silent"] for r in res) / max(1, sum(r["n_steps"] for r in res))))

In [ ]:
%%writefile /kaggle/working/stages/s6b_pdf_defs.py
# ===================== STEP 6b : PDF REPORT (template-style, one entry per 15 s) =====================
from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.lib.units import mm
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, Image, KeepTogether, PageBreak
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

NAVY, LBLUE, PALE, GREY = colors.HexColor("#2B2F77"), colors.HexColor("#CDEBFF"), colors.HexColor("#EEF7FF"), colors.HexColor("#E6E6E6")
RATING_COL = {"Excellent": "#1B8A5A", "Good": "#4C9F38", "Fair": "#D98E04", "Needs attention": "#C0392B"}

def _png(fig, dpi=160):
    b = io.BytesIO(); fig.savefig(b, format="png", dpi=dpi, bbox_inches="tight"); plt.close(fig); b.seek(0); return b
def _img(buf, width_mm):
    from PIL import Image as PI
    w, h = PI.open(buf).size; buf.seek(0); return Image(buf, width=width_mm * mm, height=width_mm * mm * h / w)

def fig_window(r, speech_classes, body_classes):
    fig, axs = plt.subplots(1, 3, figsize=(8.6, 2.1), gridspec_kw={"width_ratios": [1, 1, 0.8]})
    if r["speech_mean"] is not None: axs[0].barh(speech_classes, r["speech_mean"], color="#2B2F77")
    else: axs[0].text(.5, .5, "no speech", ha="center", va="center"); axs[0].set_xticks([]); axs[0].set_yticks([])
    axs[0].set_title("Voice emotion", fontsize=8); axs[0].set_xlim(0, 1)
    if r["body_mean"] is not None:
        top = np.argsort(r["body_mean"])[::-1][:4][::-1]; axs[1].barh([body_classes[i][:18] for i in top], r["body_mean"][top], color="#E67E22")
    axs[1].set_title("Body language (top 4)", fontsize=8); axs[1].set_xlim(0, 1)
    g = r["gate"]; xs = np.arange(len(g)); axs[2].bar(xs, g, color="#2B2F77", label="voice"); axs[2].bar(xs, 1 - g, bottom=g, color="#E67E22", label="body")
    axs[2].set_ylim(0, 1); axs[2].set_title("Fusion gate per step", fontsize=8); axs[2].set_xlabel("time-step", fontsize=7); axs[2].legend(fontsize=6, loc="upper right")
    for a in axs: a.tick_params(labelsize=7)
    plt.tight_layout(); return _png(fig)

def build_pdf_report(path, res, summary, meta, speech_classes, body_classes):
    ss = getSampleStyleSheet()
    H1 = ParagraphStyle("H1", parent=ss["Title"], textColor=NAVY, fontSize=19, alignment=0, spaceAfter=4)
    H2 = ParagraphStyle("H2", parent=ss["Heading2"], textColor=NAVY, fontSize=12.5, spaceBefore=8, spaceAfter=4)
    B = ParagraphStyle("B", parent=ss["BodyText"], fontSize=8.6, leading=11.2)
    SM = ParagraphStyle("SM", parent=B, fontSize=7.4, leading=9.4, textColor=colors.HexColor("#444444"))
    CELL = ParagraphStyle("CELL", parent=B, fontSize=7.8, leading=9.8)
    CELLB = ParagraphStyle("CELLB", parent=CELL, fontName="Helvetica-Bold")
    P = lambda t, st=B: Paragraph(t, st)
    box = lambda: [("GRID", (0, 0), (-1, -1), 0.6, NAVY), ("VALIGN", (0, 0), (-1, -1), "MIDDLE"), ("TOPPADDING", (0, 0), (-1, -1), 3), ("BOTTOMPADDING", (0, 0), (-1, -1), 3)]
    rc = lambda lbl: RATING_COL.get(lbl, "#333333")
    el = []

    el += [P("Candidate Interview Evaluation Report", H1), P("An Explainable Multimodal Interview Analysis for Speech and Body Language Recognition", SM), Spacer(1, 5)]
    items = meta["header"]                                   # list of (label, value) built from this run - changes with every video
    if len(items) % 2: items.append(("", ""))
    info = [[P(a, CELLB), P(b, CELL), P(c, CELLB), P(d, CELL)] for (a, b), (c, d) in zip(items[0::2], items[1::2])]
    t = Table(info, colWidths=[27 * mm, 63 * mm, 30 * mm, 60 * mm]); t.setStyle(TableStyle(box() + [("BACKGROUND", (0, 0), (0, -1), LBLUE), ("BACKGROUND", (2, 0), (2, -1), LBLUE)])); el += [t, Spacer(1, 7)]

    el.append(P("1. Overall result", H2))
    rows = [[P("Category", CELLB), P("Weighting", CELLB), P("What it looks at", CELLB), P("Score (0-10)", CELLB), P("Rating", CELLB)]]
    for nm, wt, desc in CATEGORIES:
        v = summary["cat"][nm]; lbl = rating_for(v) if v is not None else "n/a"
        rows.append([P(nm, CELLB), P(f"{wt:.0%}", CELL), P(desc, CELL), P("n/a" if v is None else f"{v:.1f}", CELL), P(f'<font color="{rc(lbl)}"><b>{lbl}</b></font>', CELL)])
    rows.append([P("Total score for candidate", CELLB), P("100%", CELL), P("Duration-weighted mean of all window scores", CELL), P(f"<b>{summary['overall']:.1f}</b>", CELL), P(f'<font color="{rc(summary["rating"])}"><b>{summary["rating"]}</b></font>', CELL)])
    rows.append([P("Recommendation for candidate", CELLB), P(summary["recommendation"], CELL), "", "", ""])
    t = Table(rows, colWidths=[38 * mm, 18 * mm, 78 * mm, 22 * mm, 24 * mm], repeatRows=1)
    t.setStyle(TableStyle(box() + [("BACKGROUND", (0, 0), (-1, 0), LBLUE), ("BACKGROUND", (0, -2), (-1, -2), PALE), ("BACKGROUND", (0, -1), (-1, -1), GREY), ("SPAN", (1, -1), (-1, -1))])); el += [t, Spacer(1, 5)]
    el.append(P(f"Speech was detected in {summary['speech_fraction']:.0%} of the time-steps; where there was no speech the score relies on body language alone. "
                "Scores describe observable voice and body-language signals only - they do not evaluate answer content or skills, and must be reviewed by a human interviewer.", SM))

    el += [P("2. Window-by-window evaluation (every 15 seconds)", H2)]
    rows = [[P("Window", CELLB), P("Voice emotion", CELLB), P("Body language", CELLB), P("Score", CELLB), P("Rating", CELLB), P("Explanation / comments", CELLB)]]
    for r in res:
        sp = f"{speech_classes[int(np.argmax(r['speech_mean']))]} ({r['speech_mean'].max():.0%})" if r["speech_mean"] is not None else "no speech"
        bd = f"{body_classes[int(np.argmax(r['body_mean']))]} ({r['body_mean'].max():.0%})" if r["body_mean"] is not None else "n/a"
        rows.append([P(f"{fmt_t(r['t0'])}-{fmt_t(r['t1'])}", CELLB), P(sp, CELL), P(bd, CELL), P(f"<b>{r['score']:.1f}</b>", CELL),
                     P(f'<font color="{rc(r["rating"])}"><b>{r["rating"]}</b></font>', CELL), P(r["comment"], CELL)])
    t = Table(rows, colWidths=[20 * mm, 24 * mm, 27 * mm, 12 * mm, 20 * mm, 77 * mm], repeatRows=1)
    t.setStyle(TableStyle(box() + [("BACKGROUND", (0, 0), (-1, 0), LBLUE), ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.white, PALE]), ("VALIGN", (0, 0), (-1, -1), "TOP")])); el.append(t)

    el += [PageBreak(), P("3. Detailed evidence per window", H2),
           P("For every window: a frame from the middle of the window, the average voice-emotion and body-language probabilities, and the fusion gate "
             "(how much the model relied on the voice vs the body in each 3-second step).", SM), Spacer(1, 4)]
    for r in res:
        left = []
        if r["thumb"] is not None:
            ok, jpg = cv2.imencode(".jpg", cv2.cvtColor(r["thumb"], cv2.COLOR_RGB2BGR)); left = _img(io.BytesIO(jpg.tobytes()), 44)
        head = Table([[P(f"<b>Window {r['idx']+1}: {fmt_t(r['t0'])} - {fmt_t(r['t1'])}</b>", CELL),
                       P(f'Score <b>{r["score"]:.1f}</b> / 10 - <font color="{rc(r["rating"])}"><b>{r["rating"]}</b></font>', CELL)]], colWidths=[100 * mm, 80 * mm])
        head.setStyle(TableStyle([("BACKGROUND", (0, 0), (-1, -1), LBLUE), ("BOX", (0, 0), (-1, -1), 0.6, NAVY)]))
        cats = "  |  ".join(f"{k.split(' &')[0].split(' ')[0]}: {'n/a' if v is None else v}" for k, v in r["cat"].items())
        body = Table([[left if left else "", [P(r["comment"], CELL), Spacer(1, 2), P(cats, SM)]]], colWidths=[48 * mm, 132 * mm])
        body.setStyle(TableStyle([("VALIGN", (0, 0), (-1, -1), "TOP"), ("BOX", (0, 0), (-1, -1), 0.6, NAVY)]))
        el.append(KeepTogether([head, body, _img(fig_window(r, speech_classes, body_classes), 175), Spacer(1, 6)]))

    best, worst = summary["best"], summary["worst"]
    el += [P("4. Highlights and points to review", H2)]
    for lab, ids in (("Strongest windows", best), ("Windows to review", worst)):
        el.append(P(f"<b>{lab}:</b> " + "; ".join(f"{fmt_t(res[i]['t0'])}-{fmt_t(res[i]['t1'])} (score {res[i]['score']:.1f})" for i in ids), B))
    el += [P("5. How to read this report", H2),
           P("<b>Pipeline.</b> The video is split into audio and picture. The speech model (spatial CNN + Wav2Vec2 with soft-label correction) predicts six voice emotions; "
             "the body-language model (Video-Swin RGB/DCT, skeleton transformer" + (", LaViLa" if meta.get("lavila") else "") + f") predicts {len(body_classes)} body-language classes. Both are computed on 3-second steps. "
             "A cross-attention fusion network then aligns the two streams in time, lets each modality attend to the other, and a Gated Multimodal Unit decides how much to trust voice versus body in every step.", B),
           P("<b>Scores.</b> Each class is mapped to positivity, composure and engagement; category scores combine these with the weights in section 1. Window scores are on a 0-10 scale "
             "(Excellent >= 7.5, Good >= 6, Fair >= 4.5, otherwise Needs attention).", B),
           P("<b>Limits.</b> " + meta["limits"], B)]

    def foot(c, d):
        c.saveState(); c.setFont("Helvetica", 7); c.setFillColor(colors.grey)
        c.drawString(15 * mm, 8 * mm, "Candidate Interview Evaluation Report - automated behavioural analysis, for human review"); c.drawRightString(195 * mm, 8 * mm, f"Page {d.page}"); c.restoreState()
    SimpleDocTemplate(path, pagesize=A4, leftMargin=15 * mm, rightMargin=15 * mm, topMargin=14 * mm, bottomMargin=16 * mm, title="Candidate Interview Evaluation Report").build(el, onFirstPage=foot, onLaterPages=foot)
    return path

In [ ]:
%%writefile /kaggle/working/stages/s7_report_run.py
# ===================== STEP 6 : scores, explanations and the PDF =====================
BODY_TRAITS, _auto = build_body_traits(BODY_CLASSES)
print("Body-language class -> (positivity, composure, engagement):")
for c in BODY_CLASSES: print(f"  {c:28s} {BODY_TRAITS[c]}" + ("   <- default (name not recognised; edit BODY_TRAIT_OVERRIDE)" if c in _auto else ""))

RES = analyse(WINDOWS, SEGS, SPEECH_OUT, BODY_OUT, FUSION_OUT, SPEECH_CLASSES, BODY_CLASSES, BODY_TRAITS)
for r, w in zip(RES, WINDOWS):
    ids = w["seg_ids"]; cand = [THUMBS[i] for i in ids[len(ids) // 2:] + ids[:len(ids) // 2] if THUMBS[i] is not None]
    r["thumb"] = cand[0] if cand else None
    r["comment"] = comment_for(r, SPEECH_CLASSES, BODY_CLASSES)
SUMMARY = overall_summary(RES, SPEECH_CLASSES, BODY_CLASSES)

_dr = [o.get("pose_detect_rate") for o in BODY_OUT if o.get("pose_detect_rate") is not None]
_cand = CANDIDATE_NAME or re.sub(r"[_\-]+", " ", os.path.splitext(os.path.basename(VIDEO_PATH))[0]).strip().title()
_header = [("Candidate", _cand), ("Video file", os.path.basename(VIDEO_PATH)),
           ("Duration", f"{fmt_t(DURATION)} ({DURATION:.0f} s)"), ("Video format", f"{VID_W}x{VID_H} @ {FPS:.0f} fps"),
           ("Windows analysed", f"{len(RES)} x {WINDOW_SEC} s ({len(SEGS)} steps of ~{SUBSEG_SEC} s)"), ("Speech detected", f"{SUMMARY['speech_fraction']:.0%} of the interview"),
           ("Speech model", f"{len(SPEECH_MODELS_USED)}-model ensemble (5 sessions, F/M folds) ({len(SPEECH_CLASSES)} emotions)"), ("Body-language model", f"{VCK['variant']} ({NUM_BODY} classes)"),
           ("Overall score", f"{SUMMARY['overall']:.1f} / 10 - {SUMMARY['rating']}"), ("Report generated", datetime.datetime.now().strftime("%Y-%m-%d %H:%M"))]
META = dict(header=_header, lavila="lavila" in MODS,
            limits=("The speech model was trained on acted IEMOCAP-style speech and the body-language model on a small labelled video set, so confidence on real interviews can be lower. "
                    "The fusion network has not yet been trained on paired interview data, so modality weighting follows model confidence rather than learned behaviour. "
                    "Emotion and body language are not measures of competence, honesty or fit; use this report only as one input to a human decision."))
PDF_PATH = os.path.join(OUT_DIR, REPORT_NAME)
build_pdf_report(PDF_PATH, RES, SUMMARY, META, SPEECH_CLASSES, BODY_CLASSES)

# save raw outputs too (for your own analysis / future fusion training)
np.savez_compressed(os.path.join(OUT_DIR, "interview_embeddings.npz"),
                    speech_emb=np.stack([o["emb"] for o in SPEECH_OUT]), speech_probs=np.stack([o["probs"] for o in SPEECH_OUT]), speech_silent=np.array([o["silent"] for o in SPEECH_OUT]),
                    body_emb=np.stack([o["emb"] for o in BODY_OUT]), body_probs=np.stack([o["probs"] for o in BODY_OUT]),
                    seg_t0=np.array([s["t0"] for s in SEGS]), seg_t1=np.array([s["t1"] for s in SEGS]))
json.dump(dict(overall=SUMMARY["overall"], rating=SUMMARY["rating"], categories=SUMMARY["cat"], recommendation=SUMMARY["recommendation"],
               windows=[dict(window=r["idx"] + 1, start=r["t0"], end=r["t1"], score=r["score"], rating=r["rating"], categories=r["cat"], comment=r["comment"]) for r in RES]),
          open(os.path.join(OUT_DIR, "interview_scores.json"), "w"), indent=2)
print(f"\nOverall score {SUMMARY['overall']}/10 ({SUMMARY['rating']})")
print(f"PDF report  -> {PDF_PATH}")
try:
    from IPython.display import FileLink, display; display(FileLink(PDF_PATH, result_html_prefix="Download: "))
except Exception: pass

## Launch the UI
Loads the models once (takes a few minutes), then prints the public link.

In [ ]:
# ============================ AI INTERVIEWER - GRADIO UI ============================
# Loads your models ONCE, then every uploaded video only runs inference.
import os, re, json, time, shutil, subprocess, datetime, traceback
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import gradio as gr

STAGE_DIR   = "/kaggle/working/stages"
REPORT_DIR  = "/kaggle/working/reports"; os.makedirs(REPORT_DIR, exist_ok=True)
MAX_MINUTES = 10            # longest video the UI accepts (processing time grows with length)
NS = {"__name__": "__main__", "_CACHE": {}}     # shared namespace the stage files run in (models stay alive in here)

# --- two tiny patches so models stay in GPU memory between videos (nothing else in your code is changed) ---
PATCHES = {
    "s2_speech.py": [
        ("sers = []\n    for p in SPEECH_CKPTS:",
         "sers = _CACHE.setdefault('sers', [])\n    for p in ([] if sers else SPEECH_CKPTS):"),
        ("    EMB_IDX = min(EMB_FROM",
         "    _CACHE['used'] = _CACHE.get('used') or list(SPEECH_MODELS_USED); SPEECH_MODELS_USED = list(_CACHE['used'])\n    EMB_IDX = min(EMB_FROM"),
        ("del sers; torch.cuda.empty_cache()", "torch.cuda.empty_cache()"),
    ],
    "s4_body_run.py": [
        ("for m in list(BB.values()) + [head] + ([LAV] if LAV is not None else []): m.cpu()", "pass"),
    ],
}

def run_stage(fname):
    src = open(os.path.join(STAGE_DIR, fname)).read()
    for old, new in PATCHES.get(fname, []):
        assert old in src, f"patch target not found in {fname}: {old[:50]!r}"
        src = src.replace(old, new)
    exec(compile(src, fname, "exec"), NS)

# ------------------------------ one-time start-up ------------------------------
t0 = time.time()
print(">> loading config, body-language models, report helpers ...")
for f in ["s0_config.py", "s3_body_load.py", "s6_scores_defs.py", "s6b_pdf_defs.py"]:
    run_stage(f)

PER_VIDEO = [("Splitting audio / video", "s1_split.py", 0.05),
             ("Speech emotion (10-model ensemble)", "s2_speech.py", 0.25),
             ("Body language (Swin + LaViLa + skeleton)", "s4_body_run.py", 0.75),
             ("Cross-attention fusion", "s5_fusion.py", 0.05),
             ("Scores + PDF report", "s7_report_run.py", 0.10)]

def run_pipeline(video_path, name, progress=None):
    NS["VIDEO_PATH"] = video_path
    NS["CANDIDATE_NAME"] = (name or "").strip() or None
    done = 0.0
    for label, fname, w in PER_VIDEO:
        if progress: progress(done, desc=label)
        run_stage(fname); done += w
    if progress: progress(1.0, desc="Done")

# warm-up on a 12 s clip: loads the 10 speech models + checks the whole chain before the UI opens
_wv = NS.get("VIDEO_PATH")
if _wv and os.path.exists(_wv):
    clip = "/kaggle/working/_warmup.mp4"
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-ss", "20", "-t", "12", "-i", _wv,
                    "-c:v", "libx264", "-preset", "ultrafast", "-c:a", "aac", clip])
    print(">> warm-up run on a 12 s clip ...")
    run_pipeline(clip if os.path.exists(clip) else _wv, "warmup")
else:
    print(">> no sample video attached -> warm-up skipped (the first analysis will also load the speech models, ~1 min extra)")
SKELETON_ON = NS.get("POSE_CLS") is not None
print(f">> ready in {time.time()-t0:.0f}s | skeleton branch: {'ON' if SKELETON_ON else 'OFF (neutral values - mediapipe unavailable)'}")

# ------------------------------ results -> UI ------------------------------
RATING_COLOR = {"Excellent": "#1B8A5A", "Good": "#4C9F38", "Fair": "#D98E04", "Needs attention": "#C0392B"}

def fmt_t(sec): return f"{int(sec // 60):02d}:{int(sec % 60):02d}"

def fig_scores(R):
    fig, ax = plt.subplots(figsize=(7.5, 3.2))
    xs = [(r["t0"] + r["t1"]) / 120 for r in R]; ys = [r["score"] for r in R]
    for lo, hi, c in [(0, 4.5, "#C0392B"), (4.5, 6, "#D98E04"), (6, 7.5, "#4C9F38"), (7.5, 10, "#1B8A5A")]:
        ax.axhspan(lo, hi, color=c, alpha=0.08)
    ax.plot(xs, ys, color="#2B2F77", lw=1.6, zorder=3)
    ax.scatter(xs, ys, c=[RATING_COLOR[r["rating"]] for r in R], s=46, zorder=4)
    ax.set_ylim(0, 10); ax.set_xlabel("minutes"); ax.set_ylabel("score / 10"); ax.set_title("Score per 15-second window", fontsize=10)
    ax.spines[["top", "right"]].set_visible(False); fig.tight_layout(); return fig

def fig_share(vec, names, title, color, top=None):
    fig, ax = plt.subplots(figsize=(4.6, 3.2))
    if vec is None:
        ax.text(.5, .5, "no data", ha="center", va="center"); ax.axis("off")
    else:
        idx = np.argsort(vec)[::-1][:top] if top else np.arange(len(vec))
        idx = idx[::-1]
        ax.barh([names[i] for i in idx], [float(vec[i]) for i in idx], color=color); ax.set_xlim(0, 1)
        ax.spines[["top", "right"]].set_visible(False)
    ax.set_title(title, fontsize=10); fig.tight_layout(); return fig

def format_results(ns):
    S, R = ns["SUMMARY"], ns["RES"]
    head = (f"## Overall score: {S['overall']} / 10  -  {S['rating']}\n\n{S['recommendation']}\n\n"
            f"Speech detected in **{S['speech_fraction']:.0%}** of the video | "
            f"Best windows: {', '.join(str(i + 1) for i in S['best'])} | Windows to review: {', '.join(str(i + 1) for i in S['worst'])}")
    cats = pd.DataFrame([{"Category": k, "Score (/10)": ("n/a" if v is None else v)} for k, v in S["cat"].items()])
    wins = pd.DataFrame([{"Window": f"{fmt_t(r['t0'])} - {fmt_t(r['t1'])}", "Score": r["score"], "Rating": r["rating"], "Comment": r["comment"]} for r in R])
    f1 = fig_scores(R)
    f2 = fig_share(S["speech_share"], ns["SPEECH_CLASSES"], "Voice emotion share", "#2B2F77")
    f3 = fig_share(S["body_share"], ns["BODY_CLASSES"], "Body-language share (top 8)", "#E67E22", top=8)
    return head, cats, wins, f1, f2, f3

def analyze(video, name, progress=gr.Progress()):
    if not video:
        raise gr.Error("Please upload or record a video first.")
    import cv2
    cap = cv2.VideoCapture(video); fps = cap.get(cv2.CAP_PROP_FPS) or 25.0; n = cap.get(cv2.CAP_PROP_FRAME_COUNT); cap.release()
    if n / fps > MAX_MINUTES * 60:
        raise gr.Error(f"Video is {n/fps/60:.1f} min long - the limit is {MAX_MINUTES} min. Trim it and try again.")
    t0 = time.time()
    try:
        run_pipeline(video, name, progress)
    except Exception as e:
        traceback.print_exc(); raise gr.Error(f"Pipeline failed: {type(e).__name__}: {str(e)[:300]}")
    head, cats, wins, f1, f2, f3 = format_results(NS)
    stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    pdf = os.path.join(REPORT_DIR, f"report_{stamp}.pdf"); js = os.path.join(REPORT_DIR, f"scores_{stamp}.json")
    shutil.copy(NS["PDF_PATH"], pdf); shutil.copy(os.path.join(NS["OUT_DIR"], "interview_scores.json"), js)
    head += f"\n\n*Processed in {time.time()-t0:.0f}s.*"
    return head, cats, wins, f1, f2, f3, pdf, js

NOTE = ("**Please read:** the speech model was trained on acted IEMOCAP speech and the body-language model on a small labelled video set; "
        "the fusion network is not yet trained on paired interview data. Voice and body-language signals are **not** measures of competence, "
        "honesty or fit - use this only as one input to a human decision.")

with gr.Blocks(title="AI Interviewer") as demo:
    gr.Markdown("# AI Interviewer - Speech + Body-Language Evaluation\nUpload an interview video. The app splits audio and video, "
                "runs your speech-emotion and body-language models, fuses them with cross-attention, and builds a PDF report.")
    gr.Markdown(f"Skeleton branch: **{'ON' if SKELETON_ON else 'OFF (neutral values)'}**  |  Max length: {MAX_MINUTES} min")
    with gr.Row():
        with gr.Column(scale=2):
            vid = gr.Video(label="Interview video", sources=["upload", "webcam"])
            nm = gr.Textbox(label="Candidate name (optional)", placeholder="e.g. Mr. John")
            btn = gr.Button("Analyze interview", variant="primary")
        with gr.Column(scale=3):
            out_head = gr.Markdown("Results will appear here.")
            out_cats = gr.Dataframe(label="Category scores", interactive=False)
    with gr.Tabs():
        with gr.Tab("Timeline"):
            out_wins = gr.Dataframe(label="Per 15-second window", interactive=False, wrap=True)
            out_f1 = gr.Plot(label="Score timeline")
        with gr.Tab("Voice & body"):
            with gr.Row():
                out_f2 = gr.Plot(); out_f3 = gr.Plot()
        with gr.Tab("Downloads"):
            out_pdf = gr.File(label="PDF report"); out_js = gr.File(label="Scores (JSON)")
    gr.Markdown(NOTE)
    btn.click(analyze, [vid, nm], [out_head, out_cats, out_wins, out_f1, out_f2, out_f3, out_pdf, out_js])

demo.queue(default_concurrency_limit=1)
demo.launch(share=True, debug=True)       # prints a public https://xxxx.gradio.live link; keep this cell running
